# Exp 8 - Vehicle-to-Vehicle (V2V) Communication Simulation using SUMO

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Model V2V communication opportunities based on vehicle positions and radio range.

The requested experiment mentions SUMO. SUMO is not installed in this environment, so this notebook uses a standard-library SUMO-style trace as an executable fallback. The architecture and formulas still match the V2V/SUMO workflow.

## Textbook Notes and Case Studies

### 1. Textbook Background

Vehicle-to-Vehicle communication allows vehicles to exchange information such as position, speed, heading, braking state, and hazard warnings. The objective is cooperative awareness: one vehicle can react not only to what its own sensors see, but also to what neighboring vehicles report.

SUMO is a traffic simulator used to model vehicle movement, road networks, routes, and mobility patterns. If SUMO is installed, a V2V experiment can couple mobility simulation with communication logic. In this notebook, the fallback simulation models vehicle positions and neighbor detection using Python because SUMO is not available in the current environment.

### 2. Architecture Notes

```
Vehicle Mobility Model / SUMO
        |
        v
Vehicle Position, Speed, Heading
        |
        v
Neighbor Discovery by Communication Range
        |
        v
V2V Message Exchange
        |
        v
Alert / Cooperative Awareness Decision
```

The communication layer depends on mobility. Two vehicles may be connected at one time step and disconnected later as distance changes. A valid V2V study therefore needs both network logic and movement logic.

### 3. Important Formulas

Euclidean distance between vehicles:

```
distance = sqrt((x2 - x1)^2 + (y2 - y1)^2)
```

Communication condition:

```
connected = distance <= communication_range
```

Relative speed:

```
relative_speed = speed_front - speed_rear
```

Simplified time-to-collision when rear vehicle is closing:

```
TTC = gap / (rear_speed - front_speed)
```

TTC is only meaningful under a simplified straight-line assumption. Curved roads, acceleration, sensor errors, and driver behavior require more complex models.

### 4. Classroom Case Studies

Case Study A - Sudden Braking Broadcast:
A lead vehicle brakes hard and broadcasts a warning. A following vehicle outside direct sensor range may still receive the warning through V2V and prepare earlier.

Case Study B - Blind Intersection:
Two vehicles approach an intersection with blocked line of sight. V2V messages can reveal approach speed and direction before cameras or lidar can see the other vehicle.

Case Study C - Highway Platoon:
Vehicles in a platoon exchange speed and acceleration. Communication range and message delay affect how tightly the platoon can safely maintain spacing.

### 5. Analysis Checklist

Record vehicle positions, pairwise distances, connected pairs, alerts generated, and any assumptions about range. Do not claim a real SUMO run unless SUMO is installed and the notebook connects to it. This notebook's output is a Python fallback mobility and communication model.

### 6. Source Notes

- SUMO official documentation: https://sumo.dlr.de/docs/
- Python mathematical functions used in the fallback model: https://docs.python.org/3/library/math.html


## Architecture

```text
Traffic Mobility Source
  |-- SUMO/TraCI in a full setup
  |-- standard-library trace in this notebook
          |
          v
Vehicle State
  |-- vehicle id
  |-- position
  |-- speed
          |
          v
Radio Range Model
  |-- distance between vehicles
  |-- link exists if distance <= range
          |
          v
V2V Link Graph
```

In a full SUMO run, TraCI is the interface used to read and control simulated objects online. Here, the same concept is represented by generated positions.

## Formulas and Required Theory

Vehicle position under constant-speed fallback model:

\[
x_i(t) = x_i(0) + v_i t
\]

Pairwise distance:

\[
d_{ij}(t) = |x_i(t) - x_j(t)|
\]

Communication-link rule:

\[
\text{link}_{ij}(t) =
\begin{cases}
1, & d_{ij}(t) \le R\\
0, & d_{ij}(t) > R
\end{cases}
\]

Delivery ratio:

\[
\text{delivery ratio} = \frac{\text{connected vehicle pairs}}{\text{total vehicle pairs}}
\]

## In-Lab Method

1. Define vehicles with initial positions and speeds.
2. Advance time in fixed steps.
3. Compute each vehicle position at every time step.
4. Compute pairwise distances.
5. Mark a V2V link when distance is within radio range.

In [1]:
import math

print("EXP 8 - IN-LAB V2V SIMULATION")
print("SUMO is not installed in this environment, so this notebook executes a standard-library SUMO-style mobility trace.")
vehicles = {
    "V1": {"x": 0, "speed": 15},
    "V2": {"x": 55, "speed": 13},
    "V3": {"x": 120, "speed": 16},
}
radio_range = 90
for t in range(0, 11, 2):
    positions = {v: data["x"] + data["speed"] * t for v, data in vehicles.items()}
    links = []
    for a in positions:
        for b in positions:
            if a < b and abs(positions[a] - positions[b]) <= radio_range:
                links.append(f"{a}-{b}")
    print(f"t={t:2}s positions={positions} links={links}")

EXP 8 - IN-LAB V2V SIMULATION
SUMO is not installed in this environment, so this notebook executes a standard-library SUMO-style mobility trace.
t= 0s positions={'V1': 0, 'V2': 55, 'V3': 120} links=['V1-V2', 'V2-V3']
t= 2s positions={'V1': 30, 'V2': 81, 'V3': 152} links=['V1-V2', 'V2-V3']
t= 4s positions={'V1': 60, 'V2': 107, 'V3': 184} links=['V1-V2', 'V2-V3']
t= 6s positions={'V1': 90, 'V2': 133, 'V3': 216} links=['V1-V2', 'V2-V3']
t= 8s positions={'V1': 120, 'V2': 159, 'V3': 248} links=['V1-V2', 'V2-V3']
t=10s positions={'V1': 150, 'V2': 185, 'V3': 280} links=['V1-V2']


## Post-Lab Method

The post-lab cell varies radio range and computes delivery ratio. This shows the relationship between communication range and network connectivity.

In [2]:
import math

print("EXP 8 - POST-LAB PACKET DELIVERY VS RANGE")
positions = {"V1": 150, "V2": 185, "V3": 260, "V4": 370}
for radio_range in [50, 100, 150, 250]:
    possible = 0
    delivered = 0
    for a in positions:
        for b in positions:
            if a < b:
                possible += 1
                delivered += abs(positions[a] - positions[b]) <= radio_range
    print(f"range={radio_range:3} m delivery_ratio={delivered / possible:.2f}")

EXP 8 - POST-LAB PACKET DELIVERY VS RANGE
range= 50 m delivery_ratio=0.17
range=100 m delivery_ratio=0.33
range=150 m delivery_ratio=0.67
range=250 m delivery_ratio=1.00


## What to Write in the Lab Record

- Clearly state that this executed notebook uses a SUMO-style fallback because SUMO is unavailable.
- Include the time-step link table.
- Include delivery ratio versus radio range.
- Explain how a real SUMO/TraCI implementation would replace the generated positions with simulator vehicle positions.

## References

- Eclipse SUMO project: https://eclipse.dev/sumo/
- SUMO TraCI documentation: https://sumo.dlr.de/docs/TraCI/index.html